In [15]:
import time
print(time.ctime(time.time()))

Tue Nov 19 18:01:40 2024


# Import Libraries

In [16]:
import gsw
import os
import pickle
import random
import sys
import warnings

import multiprocess as mp
import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from itertools import combinations
from pathlib import Path
from scipy.special import comb

from haversine import haversine


In [17]:
#warnings.simplefilter('ignore')

# Set paths

In [18]:
# Set paths for main directories
# Main path
path_main = Path(os.path.abspath('')).parent.parent.parent
print(path_main)
# Working path
path_work = Path(os.path.abspath(''))
print(path_work)
# Files path
path_files = os.path.join(path_work,'files')

/Users/tarry/SCIENCE/git/FaSt-SWOT
/Users/tarry/SCIENCE/git/FaSt-SWOT/code/data_analysis/drifter_DKP


# Load configuration file

In [19]:
fname = 'config_file_run_standard_leg2.pkl'
fpath = os.path.join(path_files,fname)
with open(fpath,'rb') as f:
    config_dict = pickle.load(f)

In [20]:
config_dict

{'leg': 2,
 'ndrifters': 5,
 'run_times': DatetimeIndex(['2023-05-10 06:00:00', '2023-05-10 07:00:00',
                '2023-05-10 08:00:00', '2023-05-10 09:00:00',
                '2023-05-10 10:00:00', '2023-05-10 11:00:00',
                '2023-05-10 12:00:00', '2023-05-10 13:00:00',
                '2023-05-10 14:00:00', '2023-05-10 15:00:00',
                ...
                '2023-05-14 21:00:00', '2023-05-14 22:00:00',
                '2023-05-14 23:00:00', '2023-05-15 00:00:00',
                '2023-05-15 01:00:00', '2023-05-15 02:00:00',
                '2023-05-15 03:00:00', '2023-05-15 04:00:00',
                '2023-05-15 05:00:00', '2023-05-15 06:00:00'],
               dtype='datetime64[ns]', length=121, freq='h'),
 'domain_limits': True}

In [21]:
# Drifters per cluster
npol = config_dict['ndrifters']
leg = config_dict['leg']
# Create directory to store result files
path_DKP = os.path.join(path_files,'LS',f"leg{leg}_n{npol}")
if not os.path.exists(path_DKP):
    os.mkdir(path_DKP)


# Load data

In [29]:
fname = 'drifters_DKP_Leg2.parquet'
fpath = os.path.join(path_files,fname)
data = pd.read_parquet(fpath, engine='pyarrow')

In [30]:
data

,LAT,LON,U,V,i,type
time,,,,,,
2023-04-26 10:20:00,39.646747,1.422510,-0.034984,0.053404,0,carthe
2023-04-26 10:30:00,39.647035,1.422265,-0.052946,0.081705,0,carthe
2023-04-26 10:40:00,39.647476,1.421894,-0.064663,0.101492,0,carthe
2023-04-26 10:50:00,39.648024,1.421441,-0.070641,0.113436,0,carthe
2023-04-26 11:00:00,39.648636,1.420946,-0.073206,0.120681,0,carthe
...,...,...,...,...,...,...
2023-05-25 18:10:00,39.106813,1.541232,-0.040168,0.039612,39,hereon
2023-05-25 18:20:00,39.107027,1.540953,-0.030380,0.050173,39,hereon
2023-05-25 18:30:00,39.107298,1.540742,-0.020701,0.056198,39,hereon


# Parameters of the run

In [31]:
# Time range
run_timestamps = config_dict['run_times']
T = len(run_timestamps)
# Domain limitation?
domain_limit = config_dict['domain_limits']
if domain_limit:
    #lonw = 1.2
    lonw = 0.9
    #lone = 2
    lone = 1.8
    #lats = 39.6
    lats = 39.4
    #latn = 40
    latn = 40.2


In [32]:
# Polygon splitting
nsplit = 1

# Define Class and Functions

In [33]:
class Polygon:
    __slots__ = (
        'id', 
        'drifters', 
        'lats', 
        'lons', 
        'com', 
        'us', 
        'vs', 
        'length', 
        'lengths',
        'la',
        'lb', 
        'aspect', 
        'angle', 
        'A', 
        'B')

    def __init__(self,i,comb,data):
        # initialize polygon
        self.id = i
        self.drifters = comb # indices of drifters in polygon
        self.lats = data.LAT.values
        self.lons = data.LON.values
        self.com = [np.nanmean(data.LON.values), np.nanmean(data.LAT.values)]
        self.us = data.U.values
        self.vs = data.V.values
        self.length = []
        self.lengths = []
        self.la = [] # minor axis cluster length
        self.lb = [] # major axis cluster length
        self.aspect = []
        self.angle = []
        self.A = []
        self.B = []

    def calc_lengths(self):
        """Compute polygon length scale."""
        pairs = combinations(range(len(self.lons)), 2)
        lengths = [haversine(self._point(i), self._point(j)) for i, j in pairs]
        self.length = np.sqrt(np.mean(np.array(lengths) ** 2))

    def _point(self, i):
        """Return coordinates of a point."""
        return [self.lons[i], self.lats[i]]
    
    def calc_com(self):
        """Calculate center of mass."""
        self.com = [np.nanmean(self.lons), np.nanmean(self.lats)]  

    def least_square_method(self):
        """Estimate velocity gradients using least squares."""
        import scipy.linalg as la

        n = len(self.lons)
        dlon = [
            haversine([self.lons[i], self.com[1]], self.com) * np.sign(self.lons[i] - self.com[0])
            for i in range(n)
        ]
        dlat = [
            haversine([self.com[0], self.lats[i]], self.com) * np.sign(self.lats[i] - self.com[1])
            for i in range(n)
        ]
        R = np.vstack((np.ones(n), dlon, dlat)).T
        u0, v0 = self.us[:, None], self.vs[:, None]
        self.A = la.lstsq(R, u0, cond=None)[0][1:]
        self.B = la.lstsq(R, v0, cond=None)[0][1:]

        points = np.vstack([dlon, dlat])
        cov = np.cov(points)
        eigvals, eigvecs = np.linalg.eig(cov)
        self.la = np.sqrt(np.min(eigvals))
        self.lb = np.sqrt(np.max(eigvals))
        self.aspect = np.min(eigvals) / np.max(eigvals)
        self.angle = np.arctan(eigvecs[1, 0] / eigvecs[0, 0]) * (180 / np.pi)

In [34]:
def random_combination(iterable, r):
    "Random selection from itertools.combinations(iterable, r)"
    pool = tuple(iterable)
    n = len(pool)
    indices = sorted( random.sample(range(n), r) )
    return tuple(pool[i] for i in indices)

def makePolygons(i):
    criteria3 = data_chosen.particle.isin(combs[i])
    #print(data_chosen[criteria3])
    return Polygon(i,combs[i],data_chosen[criteria3])


def calc_properties(i):
    results[i].calc_lengths()
    results[i].calc_com()
    #results[i].calc_dens()
    results[i].least_square_method() 
    return results[i]

def calc_dkp(ux, uy, vx, vy, f):
    """Compute DKP variables."""
    div = (ux + vy) / f
    vort = (vx - uy) / f
    shearing = (vx + uy) / f
    normal = (ux - vy) / f
    strain = np.sqrt(normal**2 + shearing**2)
    strain_angle = np.arctan2(normal, shearing) * (180 / np.pi)
    return div, vort, shearing, normal, strain, strain_angle

# Velocity Gradients calculation

In [35]:

# Main processing loop
start_time = datetime.now()
for t, ttime in enumerate(run_timestamps):
    step_start = datetime.now()
    criteria = (data.index == ttime) & ((data.LON.between(lonw, lone)) & (data.LAT.between(lats, latn)) if domain_limit else True)
    data_chosen = data[criteria]
    data_chosen = data_chosen.copy()
    data_chosen["particle"] = range(len(data_chosen))

    n = len(data_chosen)
    if n < npol:
        continue

    # Calculate Coriolis parameter 
    # Based on the median latitude of the drifter array
    array_lat = data_chosen.LAT.median()
    f = gsw.f(array_lat)

    # Number of possible combinations
    Nnpol = comb(n,npol,exact=True)
    print('drifters = %d, # clusters = %d' %(n,Nnpol))
    #print('length dataset: ', data_chosen.shape[0])
    
    if Nnpol<=1:
        continue

    # Define the number of combinations
    combin=[]
    for combi in combinations(np.arange(n),npol):
        combin.append(combi)

    # if True:
    #     continue

    # split combs into 4!
    iNnpol = int(Nnpol/nsplit)
    for i in range(nsplit):
        
        if i==(nsplit-1):
            combs = combin[i*iNnpol:Nnpol]
        else:
            combs = combin[i*iNnpol:(i+1)*iNnpol]
        combs = combin[i*iNnpol:(i+1)*iNnpol]
        
        results=[]
        pool = mp.Pool(mp.cpu_count())
        results = pool.map(makePolygons, range(iNnpol))
        pool.close()
        pool.join()

        pool = mp.Pool(mp.cpu_count())
        results = pool.map(calc_properties, range(iNnpol))
        pool.close()
        pool.join()

        # Read results
        As = [results[i].A for i in range(iNnpol)]
        Bs = [results[i].B for i in range(iNnpol)]
        aspects = [results[i].aspect for i in range(iNnpol)]
        angles = [results[i].angle for i in range(iNnpol)]
        lengths = [results[i].length for i in range(iNnpol)]
        com = [results[i].com for i in range(iNnpol)]

        # Make nice numpy arrays
        As = np.array(As,dtype='float').squeeze()
        Bs = np.array(Bs,dtype='float').squeeze()
        lengths = np.array(lengths,dtype='float').squeeze()
        aspects = np.array(aspects,dtype='float').squeeze()
        angles = np.array(angles,dtype='float').squeeze()
        com = np.array(com,dtype='float').squeeze()

        # Define the velocity gradients
        ux = As[:,0]
        uy = As[:,1]
        vx = Bs[:,0]
        vy = Bs[:,1]

        # Calculate DKP
        div,vort,shearing,normal,strain,strainangle = calc_dkp(ux,uy,vx,vy,f)

        # convert to pandas
        df = pd.DataFrame(
            {
            'aspect':aspects,
            'angle':angles,
            'length':lengths/1000,
            'vort':vort,
            'strain':strain,
            'divs':div,
            'normal':normal,
            'shearing':shearing,
            'strainangle':strainangle,
            'lon':com[:,0],
            'lat':com[:,1]
            }
            )#,'dens':dens,'densdiff':densdiff})

        print(f'--- DF stats:    aspect={np.round(df.aspect.median(),2)}, L={np.round(df.length.median(),2)}, V={np.round(df.vort.median(),3)}, D={np.round(df.divs.median(),3)}')

    # Save to file
    fname = f"dkp_fast-SWOT_leg{leg}_n{npol}_{t:03d}_{ttime.strftime('%Y%m%dT%H%M')}.parquet"
    fpath = os.path.join(path_DKP,fname)
    df.to_parquet(fpath, engine='pyarrow', compression='gzip') 

    step_end = datetime.now()
    print(f"Step {t+1}/{T} completed in {(step_end - step_start).seconds / 60:.2f} minutes.")


drifters = 28, # clusters = 98280
--- DF stats:    aspect=0.11, L=27.33, V=-0.011, D=-0.054
Step 1/121 completed in 0.23 minutes.
drifters = 28, # clusters = 98280
--- DF stats:    aspect=0.11, L=27.33, V=-0.006, D=-0.011
Step 2/121 completed in 0.23 minutes.
drifters = 28, # clusters = 98280
--- DF stats:    aspect=0.11, L=27.43, V=-0.02, D=0.013
Step 3/121 completed in 0.23 minutes.
drifters = 28, # clusters = 98280
--- DF stats:    aspect=0.12, L=27.56, V=-0.034, D=0.026
Step 4/121 completed in 0.23 minutes.
drifters = 28, # clusters = 98280
--- DF stats:    aspect=0.12, L=27.68, V=-0.046, D=0.016
Step 5/121 completed in 0.23 minutes.
drifters = 28, # clusters = 98280
--- DF stats:    aspect=0.13, L=27.79, V=-0.061, D=0.005
Step 6/121 completed in 0.23 minutes.
drifters = 28, # clusters = 98280
--- DF stats:    aspect=0.13, L=27.86, V=-0.073, D=-0.003
Step 7/121 completed in 0.23 minutes.
drifters = 28, # clusters = 98280
--- DF stats:    aspect=0.13, L=27.9, V=-0.082, D=0.002
Step 